### Analysis

In [ ]:
# analysis_exploration.ipynb

# Notebook header
%load_ext autoreload
%autoreload 2

import pandas as pd
from my_analysis.data_preparation import prepare_binned_spike_data, aggregate_trial_level
from my_analysis.stats import perform_test_on_dataframe_rows, permutation_anova_test

# Step 1: Load and prepare data
date = "2023-09-26"
round_no = 1
bin_size = 0.05

analysis_df = prepare_binned_spike_data(date, round_no, bin_size)
filtered_df = analysis_df[analysis_df['MonkeyGroup'] == 'Zombies']
trial_level_df = aggregate_trial_level(filtered_df)

# Step 2: Run permutation ANOVA (trial-level, by MonkeyName)
all_results = []

unique_neurons = trial_level_df['NeuronID'].unique()
for neuron_id in unique_neurons:
    neuron_df = trial_level_df[trial_level_df['NeuronID'] == neuron_id]
    grouped = neuron_df.groupby('MonkeyName')['SpikeCount'].apply(list)

    if len(grouped) < 2:
        continue

    anova_input_df = pd.DataFrame([grouped])

    results, _ = perform_test_on_dataframe_rows(
        anova_input_df,
        test_func=permutation_anova_test,
        num_permutations=1000
    )

    for result in results:
        index, f_stat, p_value = result
        all_results.append({
            'NeuronID': neuron_id,
            'F-statistic': f_stat,
            'p-value': p_value
        })

# Step 3: Process and display results
results_df = pd.DataFrame(all_results)
significant_df = results_df[results_df['p-value'] < 0.05]

print("Significant neurons (p < 0.05):")
print(significant_df)

# Step 4: Optional: Save results
results_df.to_csv('../results/permutation_anova_results.csv', index=False)
significant_df.to_csv('../results/permutation_anova_significant.csv', index=False)